<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_12_modules_stdlib/note_lesson_12_modules_stdlib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 12 — Модулі та стандартна бібліотека: кафе розкладає код на файли

В уроці 7 звіт кафе розклали на функції. Тепер каса віддає чеки з **міткою часу** (`2024-07-19 18:30`), власниця хоче звіт за будь-який місяць, а адміністратор — запускати його з термінала командою `python main.py 2024 7`.

У цьому ноутбуці ми створимо проєкт з чотирьох модулів прямо з клітинок:

| Файл | Роль |
|---|---|
| `rules.py` | мітка часу → день тижня, година → прийом їжі |
| `orders.py` | чеки з каси (`RawOrder`) і для звіту (`Order`), генерація даних |
| `report.py` | звіт: функції з уроку 7 |
| `main.py` | точка входу: `python main.py 2024 7` |

Клітинка, що починається з `%%writefile rules.py`, не виконує код, а **записує його у файл** поруч з ноутбуком. У Colab файли з'являються на панелі 📁 ліворуч.

Виконуй клітинки **зверху вниз**. Теорія — у книзі: [Урок 12. Модулі та стандартна бібліотека](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_12/).

## 🔁 Пригадай (без підглядання)

1. В уроці 10 ми писали `from collections import deque`, а в уроці 8 — `import time` і потім `time.perf_counter()`. Чому в першому випадку пишемо просто `deque(...)`, а в другому — з префіксом `time.`?
2. Які поля має `Order` у звіті кафе з уроку 7?
3. `print`, `len` і `sorted` ми ніколи не імпортували. Звідки вони беруться?

<details>
<summary>Відповіді</summary>

1. `from collections import deque` бере з модуля одне ім'я. `import time` дає лише ім'я модуля, тому до функцій звертаємося через крапку.
2. `total_bill`, `tip`, `day`, `time`, `size`.
3. Це вбудовані (built-in) функції: вони доступні завжди, без імпорту.

</details>

## 1. Модуль — це файл

Модуль — звичайний файл `.py`, його ім'я — ім'я файлу без `.py`. Почнемо з правил кафе: у них немає жодного імпорту.

In [ ]:
%%writefile rules.py
"""Правила кафе: як мітка часу перетворюється на день тижня і прийом їжі."""

DAYS = ("пн", "вт", "ср", "чт", "пт", "сб", "нд")
MONTHS = ("", "січень", "лютий", "березень", "квітень", "травень", "червень",
          "липень", "серпень", "вересень", "жовтень", "листопад", "грудень")


def day_from_timestamp(ts):
    """Короткий день тижня: datetime(2024, 7, 19, ...) -> 'пт'."""
    return DAYS[ts.weekday()]


def meal_type_from_hour(hour):
    """Прийом їжі за годиною замовлення."""
    if 11 <= hour <= 15:
        return "обід"
    if 17 <= hour <= 23:
        return "вечеря"
    return "інше"

Тепер цим файлом можна користуватися з ноутбука — так само, як зі стандартного модуля.

**Прогноз:** що надрукує кожен рядок? `datetime(2024, 7, 21)` — неділя.

In [ ]:
from datetime import datetime

import rules

print(rules.meal_type_from_hour(19))
print(rules.meal_type_from_hour(16))
print(rules.day_from_timestamp(datetime(2024, 7, 21, 12, 0)))
print(rules.MONTHS[7])

<details>
<summary>Відповідь</summary>

`вечеря`, `інше` (16 — між обідом і вечерею), `нд`, `липень`.

</details>

## 2. Три способи імпорту

| Спосіб | Як викликати |
|---|---|
| `import rules` | `rules.meal_type_from_hour(19)` |
| `from rules import meal_type_from_hour` | `meal_type_from_hour(19)` |
| `import statistics as st` | `st.mean(bills)` |

`from math import *` забирає **всі** імена модуля і може непомітно замінити твої або вбудовані. **Прогноз:** що надрукують два однакові `print`?

In [ ]:
print(pow(2, 3))

from math import *

print(pow(2, 3))

<details>
<summary>Відповідь</summary>

`8`, а потім `8.0`: `math.pow` замінила вбудовану `pow`, і нічого про це не сказала. Тому `import *` не пишуть.

</details>

### Пастка: `datetime` — і модуль, і клас

Після `import datetime` ім'я `datetime` означає **модуль**. Розкоментуй другий рядок і запусти — отримаєш `TypeError: 'module' object is not callable`. Потім закоментуй назад.

In [ ]:
import datetime
# ts = datetime(2024, 7, 19)

print(datetime.datetime(2024, 7, 19))   # так можна: модуль.клас

from datetime import datetime           # а так ім'я datetime стає класом
print(datetime(2024, 7, 19))

## 3. Модуль, що імпортує інший модуль

`orders.py` імпортує стандартну бібліотеку і наш `rules.py`. Імпорти стоять на початку, трьома групами (PEP 8): стандартна бібліотека, сторонні пакети, модулі проєкту.

In [ ]:
%%writefile orders.py
"""Чеки кафе: сирі записи з каси і перетворення на Order з уроку 7."""
import random
from datetime import datetime, timedelta
from typing import NamedTuple

from rules import day_from_timestamp, meal_type_from_hour


class RawOrder(NamedTuple):
    """Чек, як його віддає каса: з міткою часу."""
    total_bill: float
    tip: float
    size: int
    timestamp: datetime


class Order(NamedTuple):
    """Чек, як його рахує звіт з уроку 7."""
    total_bill: float
    tip: float
    day: str
    time: str
    size: int


def generate_raw_orders(n, seed=None):
    """n випадкових чеків за 2023–2024 роки, кафе працює з 9:00 до 23:00."""
    rng = random.Random(seed)
    start = datetime(2023, 1, 1)
    days = (datetime(2025, 1, 1) - start).days
    raw_orders = []
    for _ in range(n):
        timestamp = start + timedelta(days=rng.randrange(days),
                                      hours=rng.randint(9, 22),
                                      minutes=rng.randint(0, 59))
        bill = round(rng.uniform(150, 1500), 2)
        tip = round(bill * rng.uniform(0, 0.15), 2)
        raw_orders.append(RawOrder(bill, tip, rng.randint(1, 6), timestamp))
    return raw_orders


def in_month(raw, year, month):
    """Чи належить чек до вказаного місяця."""
    return raw.timestamp.year == year and raw.timestamp.month == month


def to_order(raw):
    """RawOrder -> Order: день і прийом їжі обчислюються з мітки часу."""
    return Order(
        total_bill=raw.total_bill,
        tip=raw.tip,
        day=day_from_timestamp(raw.timestamp),
        time=meal_type_from_hour(raw.timestamp.hour),
        size=raw.size,
    )

In [ ]:
from orders import RawOrder, generate_raw_orders, to_order

raw = RawOrder(540.0, 50.0, 2, datetime(2024, 7, 19, 18, 30))
print(to_order(raw))

raw_orders = generate_raw_orders(2000, seed=42)
print(len(raw_orders))
print(raw_orders[0])

## 4. Що робить `import`

1. **Кеш:** модуль уже в `sys.modules` → береться готовий, файл не виконується.
2. **Пошук:** інакше Python перебирає теки `sys.path` по черзі; перша — тека скрипта (у ноутбуці — поточна).
3. **Виконання:** файл модуля виконується зверху вниз.
4. **Збереження:** модуль потрапляє в `sys.modules`.

In [ ]:
import sys

print("rules" in sys.modules, "orders" in sys.modules)
print(sys.modules["rules"].__file__)
print(sys.path[:3])

### Побічний ефект імпорту

Коли `report.py` переносили з ноутбука, в кінець файлу потрапила перевірка з уроку 7 — прямо на верхньому рівні. Запишемо таку першу версію.

**Прогноз:** що надрукує клітинка з двома `import report`, що йде після запису файлу?

In [ ]:
%%writefile report.py
"""Звіт кафе: функції з уроку 7, тепер в окремому модулі."""
from collections import Counter

from orders import Order
from rules import DAYS


def count_by_day(orders):
    """Кількість чеків у кожен день."""
    return Counter(order.day for order in orders)


def revenue_by_day(orders):
    """Сума чеків за кожен день."""
    revenue = {}
    for order in orders:
        revenue[order.day] = revenue.get(order.day, 0) + order.total_bill
    return revenue


def best_day(revenue):
    """День з найбільшим виторгом."""
    best = None
    for day, amount in revenue.items():
        if best is None or amount > revenue[best]:
            best = day
    return best


def print_report(orders):
    """Друкує звіт кафе за списком чеків."""
    counts = count_by_day(orders)
    revenue = revenue_by_day(orders)
    for day in DAYS:
        if day in revenue:
            average = revenue[day] / counts[day]
            print(f"{day} — чеків: {counts[day]:>2}, виторг: {revenue[day]:>9.2f}, середній: {average:.2f}")
    print("Найкращий день:", best_day(revenue))
    print("За прийомом їжі:", dict(Counter(order.time for order in orders).most_common()))



orders = [
    Order(540.0, 50.0, "пт", "вечеря", 2),
    Order(320.0, 30.0, "пт", "обід", 1),
    Order(980.0, 120.0, "сб", "вечеря", 4),
]
print_report(orders)

In [ ]:
import report
print("--- другий import ---")
import report

<details>
<summary>Відповідь</summary>

Тестовий звіт з трьох чеків — один раз, під час першого імпорту: Python виконав `report.py` зверху вниз. Другий `import` нічого не друкує, бо модуль уже в `sys.modules`.

</details>

## 5. `if __name__ == "__main__":`

| Як використали файл | `__name__` всередині нього |
|---|---|
| `python report.py` | `"__main__"` |
| `import report` | `"report"` |

Ховаємо тестовий код під умову і перезаписуємо файл.

In [ ]:
%%writefile report.py
"""Звіт кафе: функції з уроку 7, тепер в окремому модулі."""
from collections import Counter

from rules import DAYS


def count_by_day(orders):
    """Кількість чеків у кожен день."""
    return Counter(order.day for order in orders)


def revenue_by_day(orders):
    """Сума чеків за кожен день."""
    revenue = {}
    for order in orders:
        revenue[order.day] = revenue.get(order.day, 0) + order.total_bill
    return revenue


def best_day(revenue):
    """День з найбільшим виторгом."""
    best = None
    for day, amount in revenue.items():
        if best is None or amount > revenue[best]:
            best = day
    return best


def print_report(orders):
    """Друкує звіт кафе за списком чеків."""
    counts = count_by_day(orders)
    revenue = revenue_by_day(orders)
    for day in DAYS:
        if day in revenue:
            average = revenue[day] / counts[day]
            print(f"{day} — чеків: {counts[day]:>2}, виторг: {revenue[day]:>9.2f}, середній: {average:.2f}")
    print("Найкращий день:", best_day(revenue))
    print("За прийомом їжі:", dict(Counter(order.time for order in orders).most_common()))


if __name__ == "__main__":
    from orders import Order

    demo = [
        Order(540.0, 50.0, "пт", "вечеря", 2),
        Order(320.0, 30.0, "пт", "обід", 1),
        Order(980.0, 120.0, "сб", "вечеря", 4),
    ]
    print_report(demo)

⚠️ **Пастка ноутбука.** `report` уже лежить у `sys.modules`, тож новий `import report` не перечитає змінений файл. Після редагування модуля в ноутбуці його треба перезавантажити через `importlib.reload` (або перезапустити ядро: Runtime → Restart).

In [ ]:
import importlib

importlib.reload(report)
print("після reload тестовий звіт не надрукувався")
print(__name__)

А запуск файлу напряму — `__name__ == "__main__"` — друкує тестовий звіт:

In [ ]:
!python report.py

## 6. `sys.argv` і точка входу

`python main.py 2024 7` → `sys.argv == ["main.py", "2024", "7"]`. **Усі аргументи — рядки**, тому `main.py` перетворює їх через `int()`. Функція `main` отримує список параметром — так її можна викликати і з ноутбука, і з тестів.

In [ ]:
%%writefile main.py
"""Звіт кафе за місяць: python main.py РІК МІСЯЦЬ"""
import calendar
import sys

from orders import generate_raw_orders, in_month, to_order
from report import print_report
from rules import MONTHS


def main(args):
    if len(args) != 3:
        print("Використання: python main.py РІК МІСЯЦЬ, наприклад: python main.py 2024 7")
        return
    year, month = int(args[1]), int(args[2])
    raw_orders = generate_raw_orders(2000, seed=42)
    orders = [to_order(raw) for raw in raw_orders if in_month(raw, year, month)]
    days_in_month = calendar.monthrange(year, month)[1]
    print(f"Звіт кафе: {MONTHS[month]} {year}, днів: {days_in_month}, чеків: {len(orders)}")
    print_report(orders)


if __name__ == "__main__":
    main(sys.argv)

In [ ]:
!python main.py 2024 7

In [ ]:
import main

main.main(["main.py"])
print()
main.main(["main.py", "2023", "2"])

## 7. Затінення: свій `calendar.py`

Адміністраторка кладе в теку графік змін і називає файл `calendar.py`. **Прогноз:** що станеться з `python main.py 2024 7`?

У самому ноутбуці затінення не побачити: справжній `calendar` уже лежить у `sys.modules` ядра. Тому запускаємо окремий процес через `!python`.

In [ ]:
%%writefile calendar.py
STAFF = {"пн": ["Оксана", "Тарас"], "сб": ["Ірина"]}

In [ ]:
!python main.py 2024 7

In [ ]:
import os
import shutil

os.remove("calendar.py")                      # ліки: перейменувати або прибрати свій файл
shutil.rmtree("__pycache__", ignore_errors=True)
print("calendar.py видалено")

<details>
<summary>Відповідь</summary>

`AttributeError: module 'calendar' has no attribute 'monthrange'`. Тека скрипта стоїть першою в `sys.path`, тож `import calendar` знайшов графік змін, а не стандартний модуль. Python 3.13 додає підказку «consider renaming … calendar.py». Після видалення файлу запусти `!python main.py 2024 7` ще раз — звіт повернеться.

</details>

## 8. Стандартна бібліотека: п'ять модулів кафе

Перш ніж писати самому чи ставити пакет через `pip install`, перевір [перелік модулів](https://docs.python.org/3/library/index.html).

### `datetime` і `timedelta`

**Прогноз** для кожного рядка, потім запусти.

In [ ]:
from datetime import datetime, timedelta

ts = datetime(2024, 7, 19, 18, 30)
print(ts.weekday())                          # 0 — понеділок
print(ts.strftime("%d.%m.%Y %H:%M"))
print(datetime.strptime("15.07.2024", "%d.%m.%Y"))
print(ts + timedelta(hours=2))
print((datetime(2025, 1, 1) - datetime(2023, 1, 1)).days)
print(datetime(2024, 7, 1) <= ts < datetime(2024, 8, 1))

<details>
<summary>Відповідь</summary>

`4` (п'ятниця), `19.07.2024 18:30`, `2024-07-15 00:00:00`, `2024-07-19 20:30:00`, `731` (2024 — високосний), `True`.

</details>

### `random` із зерном

Те саме зерно — та сама послідовність. Тому звіт за липень щоразу однаковий.

In [ ]:
import random

rng = random.Random(42)
print(rng.randint(1, 6), rng.randint(1, 6), rng.randint(1, 6))
rng = random.Random(42)
print(rng.randint(1, 6), rng.randint(1, 6), rng.randint(1, 6))

### `collections.Counter`

Замість `counts[day] = counts.get(day, 0) + 1` з уроку 6.

In [ ]:
from collections import Counter

july = [to_order(raw) for raw in raw_orders if raw.timestamp.year == 2024 and raw.timestamp.month == 7]
times = Counter(order.time for order in july)
print(times)
print(times.most_common(1))
print(times["сніданок"])

### `calendar` і `statistics`

In [ ]:
import calendar
import statistics as st

print(calendar.monthrange(2024, 2)[1], calendar.monthrange(2023, 2)[1])
print(calendar.month_name[7], repr(calendar.month_name[0]))

bills = [320, 540, 760, 980, 4500]
print(st.mean(bills), st.median(bills))

<details>
<summary>Відповідь</summary>

`29 28`; `July ''` — нумерація з 1, назви англійською (залежать від локалі, тому кафе тримає свій `MONTHS`); `1420 760` — один банкет тягне середнє вгору, медіана показує типовий чек.

</details>

## 🛠 Вправа 1. Чайові за прийомом їжі

Напиши дві функції:

- `tip_percent(order)` — частка чайових у чеку, у відсотках;
- `tips_by_time(orders)` — словник «прийом їжі → середній відсоток чайових», округлений до одного знака, через `statistics.mean`.

Коли перевірки пройдуть, перенеси функції в `report.py` і додай у кінець `print_report` рядок `print("Чайові, %:", tips_by_time(orders))`.

In [ ]:
from statistics import mean


def tip_percent(order):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return order.tip / order.total_bill * 100
    # END SOLUTION


def tips_by_time(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    groups = {}
    for order in orders:
        groups.setdefault(order.time, []).append(tip_percent(order))
    return {time: round(mean(values), 1) for time, values in groups.items()}
    # END SOLUTION


print(tips_by_time(july))
assert tips_by_time(july) == {"обід": 7.6, "вечеря": 7.3, "інше": 6.2}
assert tips_by_time([]) == {}
print("✅ Вправа 1 пройдена")

## 🛠 Вправа 2. Звіт за тиждень

Бухгалтер хоче `python week.py 2024-07-15` — звіт за сім днів, починаючи з указаної дати. Спершу напиши функцію `week_orders(raw_orders, start_text)`, яка повертає пару `(start, orders)`:

- дату розбери через `datetime.strptime(start_text, "%Y-%m-%d")`;
- кінець — `start + timedelta(days=7)`, чек підходить, якщо `start <= raw.timestamp < end`;
- у списку — вже перетворені `Order` (через `to_order`).

In [ ]:
def week_orders(raw_orders, start_text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    start = datetime.strptime(start_text, "%Y-%m-%d")
    end = start + timedelta(days=7)
    orders = [to_order(raw) for raw in raw_orders if start <= raw.timestamp < end]
    return start, orders
    # END SOLUTION


start, week = week_orders(raw_orders, "2024-07-15")
assert start == datetime(2024, 7, 15)
assert len(week) == 21
assert Counter(order.day for order in week)["вт"] == 0
print("✅ Функція працює, чеків:", len(week))

Тепер збери файл `week.py`: `rules.py`, `orders.py` і `report.py` не змінюються, `week.py` лише імпортує з них. Без аргументу — підказка, як у `main.py`; код запуску — під `if __name__ == "__main__":`. Очікуваний перший рядок:

```text
Звіт кафе: тиждень з 2024-07-15 до 2024-07-21, чеків: 21
```

In [ ]:
%%writefile week.py
# YOUR CODE HERE
# BEGIN SOLUTION
"""Звіт кафе за тиждень: python week.py РРРР-ММ-ДД"""
import sys
from datetime import datetime, timedelta

from orders import generate_raw_orders, to_order
from report import print_report


def main(args):
    if len(args) != 2:
        print("Використання: python week.py РРРР-ММ-ДД, наприклад: python week.py 2024-07-15")
        return
    start = datetime.strptime(args[1], "%Y-%m-%d")
    end = start + timedelta(days=7)
    raw_orders = generate_raw_orders(2000, seed=42)
    orders = [to_order(raw) for raw in raw_orders if start <= raw.timestamp < end]
    last_day = end - timedelta(days=1)
    print(f"Звіт кафе: тиждень з {start.date()} до {last_day.date()}, чеків: {len(orders)}")
    print_report(orders)


if __name__ == "__main__":
    main(sys.argv)
# END SOLUTION

In [ ]:
!python week.py 2024-07-15

In [ ]:
import contextlib
import io

import week

importlib.reload(week)
out = io.StringIO()
with contextlib.redirect_stdout(out):
    week.main(["week.py", "2024-07-15"])
lines = out.getvalue().splitlines()
assert lines[0] == "Звіт кафе: тиждень з 2024-07-15 до 2024-07-21, чеків: 21", lines[0]
assert lines[-2] == "Найкращий день: ср"
print("✅ Вправа 2 пройдена")

## ✅ Самоперевірка

1. Чим `import rules` відрізняється від `from rules import meal_type_from_hour` з погляду виклику функції?
2. У `report.py` на верхньому рівні стоїть `print("звіт завантажено")`. Скільки разів він спрацює, якщо `report` імпортують двічі?
3. Чому після `%%writefile report.py` у ноутбуці потрібен `importlib.reload(report)`?
4. Поруч з `main.py` лежить твій `random.py`. Що станеться з `orders.py`?
5. Що надрукує `print(type(sys.argv[1]))` для `python main.py 2024 7`?
6. Чому `generate_raw_orders` створює `random.Random(seed)`, а не викликає `random.seed(seed)`?

<details>
<summary>Відповіді</summary>

1. `rules.meal_type_from_hour(19)` проти `meal_type_from_hour(19)`; у другому випадку ім'я `rules` у програмі не з'являється.
2. Один раз: другий `import` бере модуль із `sys.modules`.
3. Модуль уже в `sys.modules`, і `import` не перечитує файл. `reload` виконує його заново.
4. Python завантажить твій `random.py` замість стандартного, і `random.Random` впаде з `AttributeError`.
5. `<class 'str'>`.
6. `random.seed` змінює спільний генератор усієї програми (побічний ефект), а власний `Random(seed)` належить лише функції.

</details>

### Шпаргалка

```python
import rules                              # rules.DAYS
from rules import meal_type_from_hour     # meal_type_from_hour(19)
import statistics as st                   # st.mean(...)

if __name__ == "__main__":                # лише при python file.py
    main(sys.argv)                        # sys.argv — список рядків

import importlib; importlib.reload(rules) # у ноутбуці після зміни файлу

from datetime import datetime, timedelta
ts.weekday(); ts.strftime("%d.%m.%Y"); datetime.strptime(text, "%Y-%m-%d")
ts + timedelta(days=7); start <= ts < end

from collections import Counter           # Counter(values).most_common(3)
import calendar                           # calendar.monthrange(2024, 2)[1] -> 29
import random                             # random.Random(42) — відтворювано
```

## Далі

- **Урок 13 — Винятки.** `python main.py 2024 липень` падає з `ValueError`, а `python main.py 2024 13` — з помилкою `calendar`. Навчимося перехоплювати такі помилки й відповідати людині зрозуміло.
- **Урок 14 — Файли й JSON.** Чеки прийдуть з файлу, і `datetime.strptime` знадобиться знову.
- Готовий проєкт з тестами — тека [`cafe_report/`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/tree/main/module_1/lessons/lesson_12_modules_stdlib/cafe_report).